In [ ]:
import pandas as pd
import re

# Step 1: List all JSON files
files = [
    "alfar_part1.json",
    "alfar_part2.json",
    "alfar_part3.json",
    "alfar_part4.json"
]


In [3]:
df_list = []

In [4]:
def extract_audit_fields(audits):
    result = {}

    # Check if the audit column contains a list
    if isinstance(audits, list):
        for i, audit in enumerate(audits, start=1):

            # Check if each audit record is a dictionary
            if isinstance(audit, dict):
                result[f"audit{i}_date"] = audit.get("auditDate", {}).get("$date")
                result[f"audit{i}_newQty"] = audit.get("newQty")

        # Step 4: Get the latest audit quantity
        qty_keys = [k for k in result.keys() if re.match(r"audit\d+_newQty", k)]

        if qty_keys:
            last_key = sorted(qty_keys, key=lambda x: int(re.findall(r"\d+", x)[0]))[-1]
            result["latestQty"] = result[last_key]

    return pd.Series(result)

In [5]:
def get_latest_audit_date(audits):
    if isinstance(audits, list) and len(audits) > 0:
        last_audit = audits[-1]

        if isinstance(last_audit, dict):
            return last_audit.get("auditDate", {}).get("$date")

    return None

In [6]:
for file in files:
    print("Processing:", file)

    # Load the JSON file
    data = pd.read_json(file)

    # Step 7: Extract audit columns
    audit_cols = data["audit"].apply(extract_audit_fields)

    # Step 8: Get the latest audit date
    data["latestAuditDate"] = data["audit"].apply(get_latest_audit_date)

    # Step 9: Merge extracted audit columns back to the data
    data = pd.concat([data, audit_cols], axis=1)

    # Step 10: Keep only important columns
    data = data[
        [
            "_id",
            "productName",
            "qty",
            "branch",
            "sku",
            "category",
            "type",
            "latestQty",
            "latestAuditDate"
        ]
    ]

Processing: alfar_part1.json
Processing: alfar_part2.json
Processing: alfar_part3.json
Processing: alfar_part4.json


In [7]:
df_list.append(data)

In [8]:
df_final = pd.concat(df_list, ignore_index=True)


In [9]:
def assign_priority(qty):
    if qty < 0:
        return "High"
    elif qty == 0:
        return "Medium"
    else:
        return "Low"

df_final["audit_priority"] = df_final["qty"].apply(assign_priority)


In [10]:
df_final.to_csv("alfar_priority_audit_list_updated.csv", index=False)


In [12]:
print("Total rows:", df_final.shape)

Total rows: (89320, 10)


In [ ]:
#Simple Logic (Easy to Understand)

#We used the qty (quantity) of each product.

#Rule:
#If the quantity is negative → High Priority
#If the quantity is zero → Medium Priority
#If the quantity is positive → Low Priority

#High Priority (qty < 0)
#means the system says you have less than zero stock
#This is an error
#needs to be checked immediately

#Medium Priority (qty = 0)
#This means the product is out of stock
#But sometimes, the item might still be physically in the store
#So it needs verification

#Low Priority (qty > 0)
#This means the product still has stock
#No urgent problem